# Notebook 11 — Ablation and sensitivity sweeps

**ClinicalShield v2 · Revision pipeline · GPU · 2–3 hours**

## Four questions v1 answered by assertion

**R3-Q7** — *"What is the justification for using a 70/15/15 train-validation-test split, and
was the chosen proportion evaluated against alternative splitting strategies?"*

**R3-Q8** — *"Why was the best validation accuracy selected for checkpointing while early
stopping was based on validation loss? Would selecting the checkpoint based on validation
loss produce different results?"*

**R3-Q9** — *"What is the rationale for selecting an early-stopping patience value of 3, and
was the sensitivity of model performance to this parameter investigated?"*

**R3 report** — *"provide an ablation or comparative study to show whether decoding the
content before detection improves attack detection."*

Each is answered by measurement here, one factor at a time with everything else held at the
NB05 baseline.

## The yardstick

Every arm is compared against **the baseline's own seed spread**. A configuration is reported
as sensitive only where its change exceeds the variation two seeds produce on identical
settings.

Without that reference, any sweep produces differences and it is tempting to read noise as
effect. This is the same discipline Reviewer 3 was asking for when they requested variance
measures in the first place.

## A methodological problem worth reporting

Changing the split ratio re-partitions the data — and NB02 assigns payload templates *per
partition* to keep the pools disjoint. A chunk moved to a new partition therefore carries a
template from the wrong pool.

The obvious fix, dropping those chunks, loses **23%** of the data and loses *different*
amounts for different ratios. The comparison would then measure sample size as much as split
ratio.

Instead the dataset is **regenerated** for each ratio, as NB02 would: the pre-injection host
is recovered from the recorded span, and a fresh payload is drawn from the new partition's
pool. Every chunk is retained, and both group and template disjointness are re-verified per
arm.

## Runtime
GPU. Roughly 2–3 hours for the full sweep. Each arm trains from scratch; no checkpoints are
reused.

In [ ]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    "SEED": 42,
    "MODEL": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "MAX_LEN": 512,
    "BATCH_SIZE": 16,
    "LR": 2e-5,
    "WEIGHT_DECAY": 0.01,
    "EPOCHS": 5,
    "CLASS_WEIGHTS": {0: 1.0, 1: 3.0},
    "GRAD_CLIP": 1.0,

    # Baseline configuration from NB05, held fixed while one factor varies.
    "BASE": {"split": (0.70, 0.15, 0.15), "patience": 3,
             "checkpoint_on": "loss", "apply_module1": True},

    # R3-Q7: "What is the justification for using a 70/15/15 split, and was the
    # chosen proportion evaluated against alternative splitting strategies?"
    "SPLIT_SWEEP": [(0.70, 0.15, 0.15), (0.80, 0.10, 0.10), (0.60, 0.20, 0.20)],

    # R3-Q9: "What is the rationale for an early-stopping patience of 3, and was
    # sensitivity to this parameter investigated?"
    "PATIENCE_SWEEP": [2, 3, 5],

    # R3-Q8: "Why was best validation accuracy selected for checkpointing while
    # early stopping was based on validation loss?"
    "CHECKPOINT_SWEEP": ["loss", "accuracy"],

    # R3 report: "provide an ablation to show whether decoding the content before
    # detection improves attack detection."
    "MODULE1_SWEEP": [True, False],

    # Two seeds per configuration: enough to separate a real effect from seed
    # noise without a full five-seed run for every arm.
    "SWEEP_SEEDS": [13, 42],

    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
SEED = CONFIG["SEED"]

n_runs = (len(CONFIG["SPLIT_SWEEP"]) + len(CONFIG["PATIENCE_SWEEP"])
          + len(CONFIG["CHECKPOINT_SWEEP"]) + len(CONFIG["MODULE1_SWEEP"]) - 3)
print("sweeps:")
print("  split ratio      : %s   (R3-Q7)" % [f"{a:.0%}/{b:.0%}/{c:.0%}"
                                             for a, b, c in CONFIG["SPLIT_SWEEP"]])
print("  patience         : %s        (R3-Q9)" % CONFIG["PATIENCE_SWEEP"])
print("  checkpoint on    : %s   (R3-Q8)" % CONFIG["CHECKPOINT_SWEEP"])
print("  Module 1 applied : %s   (R3 report)" % CONFIG["MODULE1_SWEEP"])
print("")
print("  %d configurations x %d seeds = %d training runs"
      % (n_runs + 1, len(CONFIG["SWEEP_SEEDS"]), (n_runs + 1)*len(CONFIG["SWEEP_SEEDS"])))
print("  the baseline arm is shared across sweeps rather than retrained per sweep")


In [ ]:
# ── Cell 2 · Environment and inputs ───────────────────────────────────────────
# NB11 retrains PubMedBERT across sweep arms, so it needs the dataset and
# Module 1 only. No checkpoints are loaded; each arm trains from scratch.
import sys, os, json, random, hashlib, subprocess, shutil, time, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "results": ROOT/"data"/"results", "src": ROOT/"src"/"clinicalshield",
        "ckpt": ROOT/"checkpoints"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

import re
for pkg, mod in [("transformers","transformers"), ("torch","torch"),
                 ("scikit-learn","sklearn"), ("unidecode","unidecode"),
                 ("datasets","datasets")]:
    try:
        __import__(mod)
    except ImportError:
        print("installing", pkg, "...")
        subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

REQUIRED_FIELDS = {"split", "template_id"}

def _fields_of(path):
    with open(path, encoding="utf-8") as f:
        return set(json.loads(f.readline()).keys())

def restore_fresh(rel_path, label):
    candidates = []
    local = ROOT / rel_path
    if local.exists():
        candidates.append((local, "repo"))
    if DRIVE_ROOT and (DRIVE_ROOT / rel_path).exists():
        candidates.append((DRIVE_ROOT / rel_path, "drive"))
    if not candidates:
        raise FileNotFoundError("\n%s not found in repo or Drive. Run NB02/NB03." % label)
    for path, src in candidates:
        missing = REQUIRED_FIELDS - _fields_of(path)
        if not missing:
            if src == "drive":
                local.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, local)
            return local, src
        print("  stale copy ignored (%s): missing %s" % (src, sorted(missing)))
    raise RuntimeError("\nEvery copy of %s predates the template-disjoint fix." % label)

p_ds, src_ds = restore_fresh("data/dataset/attack_dataset_split.jsonl",
                             "attack_dataset_split.jsonl")
dataset = read_jsonl(p_ds)

print("")
print("dataset from %s  %d chunks" % (src_ds, len(dataset)))
for s in ["train","val","test"]:
    rows = [d for d in dataset if d["split"] == s]
    print("  %-6s %5d chunks  %4d adversarial"
          % (s, len(rows), sum(d["label"] for d in rows)))

# Module 3's override markers are reused by the sanitiser, so they are imported
# from the exported module rather than duplicated here.
m3_path = ROOT / "src" / "clinicalshield" / "module3.py"
if m3_path.exists():
    import importlib.util as _il
    _sp = _il.spec_from_file_location("cs_m3", m3_path)
    _m3 = _il.module_from_spec(_sp); _sp.loader.exec_module(_m3)
    CONFIG["OVERRIDE_MARKERS"] = list(_m3.OVERRIDE_MARKERS)
    print("")
    print("override markers imported from module3.py: %d"
          % len(CONFIG["OVERRIDE_MARKERS"]))
else:
    CONFIG["OVERRIDE_MARKERS"] = [
        "ignore previous", "ignore all previous", "ignore the above",
        "disregard the", "disregard all", "system override", "new instruction",
        "override:", "admin:", "[admin]", "[config]", "[system]",
        "forget everything", "forget the", "prior context is void",
        "end of document", "attention model", "do not follow",
        "your reply must", "you must instead", "instead output",
    ]
    print("")
    print("module3.py not found; using built-in marker list (%d)"
          % len(CONFIG["OVERRIDE_MARKERS"]))

# Module 1 ingestion. Module 2 was trained on decoded text (NB05,
# APPLY_MODULE1=True), so scoring raw text would leave encoded payloads
# obfuscated -- that discrepancy cost 18 points of detection in NB07.
m1_path = ROOT / "src" / "clinicalshield" / "module1.py"
if not m1_path.exists() and DRIVE_ROOT:
    _s = DRIVE_ROOT / "src" / "clinicalshield" / "module1.py"
    if _s.exists():
        m1_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(_s, m1_path)
if m1_path.exists():
    import importlib.util as _il
    _sp = _il.spec_from_file_location("cs_m1", m1_path)
    _m1 = _il.module_from_spec(_sp); _sp.loader.exec_module(_m1)
    ingest = lambda t: _m1.module1(t)["decoded"]
    print("Module 1 loaded: documents decoded before classification")
else:
    ingest = lambda t: t
    print("WARNING: module1.py not found; encoded payloads will not be decoded")

# Module 4, for the sanitised condition
m4_path = ROOT / "src" / "clinicalshield" / "module4.py"
if not m4_path.exists() and DRIVE_ROOT:
    _s4 = DRIVE_ROOT / "src" / "clinicalshield" / "module4.py"
    if _s4.exists():
        shutil.copy2(_s4, m4_path)
if not m4_path.exists():
    raise FileNotFoundError("module4.py not found -- run NB07 first")
_sp4 = _il.spec_from_file_location("cs_m4", m4_path)
_m4 = _il.module_from_spec(_sp4); _sp4.loader.exec_module(_m4)
sanitise = _m4.sanitise
segments = _m4.segments
extract_entities = _m4.extract_entities
print("Module 4 loaded for the sanitised condition")

# Module 2, for the sentence scores Module 4 depends on
import torch
from transformers import (AutoTokenizer as _AT2,
                          AutoModelForSequenceClassification as _AM2)
_DEV = "cuda" if torch.cuda.is_available() else "cpu"
_ck = ROOT / "checkpoints" / ("module2_seed%d" % CONFIG["M2_SEED"])
if not _ck.exists() and DRIVE_ROOT:
    _src = DRIVE_ROOT / "checkpoints" / ("module2_seed%d" % CONFIG["M2_SEED"])
    if _src.exists():
        shutil.copytree(_src, _ck, dirs_exist_ok=True)
if not _ck.exists():
    raise FileNotFoundError("module2 checkpoint not found -- run NB05 first")
_m2tok = _AT2.from_pretrained(_ck)
_m2mdl = _AM2.from_pretrained(_ck).to(_DEV).eval()

@torch.no_grad()
def _m2_prob(texts, batch=64):
    out = []
    for i in range(0, len(texts), batch):
        enc = _m2tok(texts[i:i+batch], truncation=True, padding=True,
                     max_length=512, return_tensors="pt").to(_DEV)
        out.extend(torch.softmax(_m2mdl(**enc).logits, dim=-1)[:, 1]
                   .cpu().numpy().tolist())
    return out

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE != "cuda":
    print("WARNING: no GPU. This notebook trains many models and is impractical on CPU.")

import base64, urllib.parse


## Re-splitting

The template library is recovered from the dataset itself: each payload is a template with
one drug name and one multiplier substituted, so reversing those substitutions returns the
template. That avoids carrying a second copy of the library and keeps this notebook
consistent with whatever NB02 actually produced.

Both leakage checks are asserted per arm, not assumed.

In [ ]:
# ── Cell 3 · Re-splitting: regenerate payloads, do not drop chunks ────────────
# Changing the ratio means re-partitioning, and NB02 assigns payload templates
# AFTER the split so the pools stay disjoint. A chunk moved to a new partition
# therefore carries a template belonging to the wrong pool.
#
# Dropping those chunks was the first approach and it fails as an experiment:
# 23% are lost, and different ratios lose different amounts, so the comparison
# measures sample size as much as split ratio.
#
# Instead the dataset is REGENERATED per ratio, as NB02 would. span_start and
# span_end were recorded, so the pre-injection host is recoverable exactly, and
# a fresh payload is drawn from the new partition's pool.

TEMPLATE_SPLIT = {"train": slice(0, 6), "val": slice(6, 8), "test": slice(8, 10)}
MULTIPLIERS = ["two", "three", "four", "five", "ten"]

def authentic_host(d):
    if d["label"] == 0:
        return d["text"]
    s0, e0 = d.get("span_start"), d.get("span_end")
    if s0 is None:
        return d["text"]
    return re.sub(r"\s+", " ", (d["text"][:s0] + d["text"][e0:]).strip())

# Recover the template library from the dataset itself: each payload is a
# template with one drug name and one multiplier substituted, so reversing those
# gives back the template. Keyed by vector/category/index, exactly as NB02.
def to_template(payload, drug):
    t = payload.replace(drug, "{drug}")
    return re.sub(r"\b(two|three|four|five|ten)\b", "{mult}", t)

TEMPLATES = {}
for d in dataset:
    if d["label"] == 1 and d.get("template_id") and d.get("encoding") == "plaintext":
        TEMPLATES.setdefault(d["template_id"], to_template(d["payload"], d["drug"]))

by_pool = defaultdict(list)
for tid, tpl in TEMPLATES.items():
    vec, cat, idx = tid.rsplit("/", 2)[0], tid.rsplit("/", 2)[1], int(tid.rsplit("/", 2)[2])
    by_pool[(vec, cat)].append((idx, tid, tpl))
for k in by_pool:
    by_pool[k].sort()

print("template library recovered from the dataset")
print("  templates : %d" % len(TEMPLATES))
print("  pools     : %d (vector x category)" % len(by_pool))
print("  per pool  : %s" % sorted({len(v) for v in by_pool.values()}))

LEET = str.maketrans({"a":"4","e":"3","i":"1","o":"0","s":"5","t":"7"})
ENCODERS = {
    "plaintext": lambda s: s,
    "base64":    lambda s: base64.b64encode(s.encode()).decode(),
    "hex":       lambda s: "".join("\\x%02x" % b for b in s.encode()),
    "unicode":   lambda s: "".join("\\u%04x" % ord(c) for c in s),
    "url":       lambda s: urllib.parse.quote(s),
    "leet":      lambda s: s.translate(LEET),
}

def inject(host, payload, rng):
    hw, pw = host.split(), payload.split()
    hw = hw[:max(len(hw) - len(pw), 30)]
    pos = rng.randint(0, len(hw))
    before, after = " ".join(hw[:pos]), " ".join(hw[pos:])
    parts = [p for p in [before, payload, after] if p]
    text = " ".join(parts)
    s0 = len(before) + (1 if before else 0)
    return text, s0, s0 + len(payload)

def regenerate(rows, ratios, seed):
    """Split at the given ratios, then draw payloads from each partition's pool."""
    rng = random.Random(seed)
    groups = sorted({r["group_id"] for r in rows})
    rng.shuffle(groups)
    n = len(groups)
    n_tr, n_va = int(round(n*ratios[0])), int(round(n*ratios[1]))
    assign = {g: ("train" if i < n_tr else ("val" if i < n_tr+n_va else "test"))
              for i, g in enumerate(groups)}

    out = []
    for r in rows:
        sp = assign[r["group_id"]]
        host = authentic_host(r)
        if r["label"] == 0:
            out.append(dict(r, split=sp, text=host,
                            word_count=len(host.split())))
            continue
        pool = [(i, tid, tpl) for i, tid, tpl in by_pool[(r["vector"], r["category"])]
                if TEMPLATE_SPLIT[sp].start <= i < TEMPLATE_SPLIT[sp].stop]
        if not pool:
            out.append(dict(r, split=sp))
            continue
        _i, tid, tpl = rng.choice(pool)
        pay = tpl.format(drug=r["drug"], mult=rng.choice(MULTIPLIERS))
        enc = r.get("encoding") or "plaintext"
        pay_enc = ENCODERS.get(enc, ENCODERS["plaintext"])(pay)
        text, s0, e0 = inject(host, pay_enc, rng)
        out.append(dict(r, split=sp, text=text, payload=pay_enc,
                        template_id=tid, span_start=s0, span_end=e0,
                        word_count=len(text.split())))
    return out

def verify_split(rows, label):
    g2s, t2s = defaultdict(set), defaultdict(set)
    for r in rows:
        g2s[r["group_id"]].add(r["split"])
        if r.get("template_id"):
            t2s[r["template_id"]].add(r["split"])
    gs = sum(1 for v in g2s.values() if len(v) > 1)
    ts = sum(1 for v in t2s.values() if len(v) > 1)
    c = Counter(r["split"] for r in rows)
    adv = {s: sum(r["label"] for r in rows if r["split"] == s)
           for s in ["train", "val", "test"]}
    print("  %-10s train %5d(%4d adv)  val %4d(%3d)  test %4d(%3d)  "
          "leak: group %d template %d"
          % (label, c["train"], adv["train"], c["val"], adv["val"],
             c["test"], adv["test"], gs, ts))
    assert gs == 0 and ts == 0, "leakage in %s" % label

print("")
print("SPLIT RATIO SWEEP — regenerated per ratio, no chunks dropped")
split_sets = {}
for ratios in CONFIG["SPLIT_SWEEP"]:
    lab = "%.0f/%.0f/%.0f" % (ratios[0]*100, ratios[1]*100, ratios[2]*100)
    rows = regenerate(dataset, ratios, SEED)
    assert len(rows) == len(dataset), "chunk count changed"
    split_sets[lab] = rows
    verify_split(rows, lab)


## Training harness

One factor varies; the rest hold at the baseline. The checkpoint criterion is a parameter
here rather than fixed, which is what makes R3-Q8 answerable: v1 stopped on validation loss
but saved on validation accuracy, so the model it kept was not the model its stopping rule
had judged best.

In [ ]:
# ── Cell 4 · Training harness ─────────────────────────────────────────────────
# One factor varies at a time; everything else holds at the NB05 baseline.

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL"])

class ChunkDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        e = tokenizer(r["model_input"], truncation=True, padding="max_length",
                      max_length=CONFIG["MAX_LEN"], return_tensors="pt")
        return {"input_ids": e["input_ids"].squeeze(0),
                "attention_mask": e["attention_mask"].squeeze(0),
                "labels": torch.tensor(r["label"], dtype=torch.long)}

def loader(rows, shuffle, seed=None):
    g = torch.Generator()
    if seed is not None: g.manual_seed(seed)
    return DataLoader(ChunkDS(rows), batch_size=CONFIG["BATCH_SIZE"], shuffle=shuffle,
                      generator=g if shuffle else None, num_workers=2,
                      pin_memory=(DEVICE == "cuda"))

def set_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); probs, preds, ys = [], [], []
    for b in dl:
        out = model(input_ids=b["input_ids"].to(DEVICE),
                    attention_mask=b["attention_mask"].to(DEVICE)).logits
        probs.extend(torch.softmax(out, -1)[:, 1].cpu().numpy())
        preds.extend(out.argmax(-1).cpu().numpy())
        ys.extend(b["labels"].numpy())
    ys, preds, probs = np.array(ys), np.array(preds), np.array(probs)
    pr, rc, f1, _ = precision_recall_fscore_support(ys, preds, average="binary",
                                                    zero_division=0)
    tn, fp, fn, tp = confusion_matrix(ys, preds, labels=[0, 1]).ravel()
    return {"accuracy": float(accuracy_score(ys, preds)), "precision": float(pr),
            "recall": float(rc), "f1": float(f1),
            "auc": float(roc_auc_score(ys, probs)) if len(set(ys)) > 1 else float("nan"),
            "fpr": float(fp/(fp+tn)) if (fp+tn) else 0.0,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def run_config(rows, seed, patience, checkpoint_on, apply_m1):
    """Train one configuration and return held-out metrics."""
    for r in rows:
        r["model_input"] = ingest(r["text"]) if apply_m1 else r["text"]
    parts = {s: [r for r in rows if r["split"] == s] for s in ["train","val","test"]}

    set_seeds(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["MODEL"], num_labels=2).to(DEVICE)
    tr, va = loader(parts["train"], True, seed), loader(parts["val"], False)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"],
                            weight_decay=CONFIG["WEIGHT_DECAY"])
    w = torch.tensor([CONFIG["CLASS_WEIGHTS"][0], CONFIG["CLASS_WEIGHTS"][1]],
                     dtype=torch.float).to(DEVICE)
    lossf = torch.nn.CrossEntropyLoss(weight=w)

    best, best_state, bad = (float("inf") if checkpoint_on == "loss" else -1.0), None, 0
    for ep in range(1, CONFIG["EPOCHS"]+1):
        model.train()
        for b in tr:
            opt.zero_grad()
            out = model(input_ids=b["input_ids"].to(DEVICE),
                        attention_mask=b["attention_mask"].to(DEVICE)).logits
            loss = lossf(out, b["labels"].to(DEVICE)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP"])
            opt.step()
        model.eval(); vl = 0.0
        with torch.no_grad():
            for b in va:
                o = model(input_ids=b["input_ids"].to(DEVICE),
                          attention_mask=b["attention_mask"].to(DEVICE)).logits
                vl += lossf(o, b["labels"].to(DEVICE)).item()
        vl /= len(va)
        vm = evaluate(model, va)
        score = vl if checkpoint_on == "loss" else vm["accuracy"]
        better = (score < best - 1e-5) if checkpoint_on == "loss" else (score > best + 1e-5)
        if better:
            best, bad = score, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state: model.load_state_dict(best_state)
    tm = evaluate(model, loader(parts["test"], False))
    tm["epochs"] = ep
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return tm

print("harness ready. one factor varies per arm; the rest hold at the NB05 baseline:")
print("  %s" % CONFIG["BASE"])


## Running the sweeps

The baseline arm is trained once and shared across all four sweeps rather than retrained for
each, which saves six runs without weakening any comparison.

In [ ]:
# ── Cell 5 · Run the sweeps ───────────────────────────────────────────────────
B = CONFIG["BASE"]
base_lab = "%.0f/%.0f/%.0f" % tuple(x*100 for x in B["split"])
arms = []

# baseline, shared across all four sweeps rather than retrained per sweep
arms.append(("baseline", "split=%s patience=%d ckpt=%s m1=%s"
             % (base_lab, B["patience"], B["checkpoint_on"], B["apply_module1"]),
             split_sets[base_lab], B["patience"], B["checkpoint_on"], B["apply_module1"]))

for ratios in CONFIG["SPLIT_SWEEP"]:                      # R3-Q7
    lab = "%.0f/%.0f/%.0f" % tuple(x*100 for x in ratios)
    if lab == base_lab: continue
    arms.append(("split", "split=%s" % lab, split_sets[lab],
                 B["patience"], B["checkpoint_on"], B["apply_module1"]))

for p in CONFIG["PATIENCE_SWEEP"]:                        # R3-Q9
    if p == B["patience"]: continue
    arms.append(("patience", "patience=%d" % p, split_sets[base_lab],
                 p, B["checkpoint_on"], B["apply_module1"]))

for ck in CONFIG["CHECKPOINT_SWEEP"]:                     # R3-Q8
    if ck == B["checkpoint_on"]: continue
    arms.append(("checkpoint", "checkpoint_on=%s" % ck, split_sets[base_lab],
                 B["patience"], ck, B["apply_module1"]))

for m1 in CONFIG["MODULE1_SWEEP"]:                        # R3 report
    if m1 == B["apply_module1"]: continue
    arms.append(("module1", "module1=%s" % m1, split_sets[base_lab],
                 B["patience"], B["checkpoint_on"], m1))

print("%d arms x %d seeds = %d runs" % (len(arms), len(CONFIG["SWEEP_SEEDS"]),
                                        len(arms)*len(CONFIG["SWEEP_SEEDS"])))
print("")

sweep_results = defaultdict(list)
t_all = time.time()
for sweep, label, rows, patience, ck, m1 in arms:
    for seed in CONFIG["SWEEP_SEEDS"]:
        t0 = time.time()
        m = run_config(rows, seed, patience, ck, m1)
        m.update({"sweep": sweep, "label": label, "seed": seed,
                  "minutes": (time.time()-t0)/60})
        sweep_results[label].append(m)
        print("  %-24s seed %-6d acc %.4f  rec %.4f  fpr %.4f  ep %d  %.1f min"
              % (label, seed, m["accuracy"], m["recall"], m["fpr"],
                 m["epochs"], m["minutes"]))
print("")
print("all arms complete in %.1f min" % ((time.time()-t_all)/60))


## Results

Each arm reported with its delta against the baseline, flagged where that delta exceeds the
baseline's seed spread.

An arm showing **no** effect is as useful as one that does: it lets the paper say a choice was
tested and did not matter, which is a stronger answer to Reviewer 3 than a justification
offered after the fact.

In [ ]:
# ── Cell 6 · Results, with the baseline's own seed spread as the yardstick ────
# A sweep arm only "matters" if it moves the metric further than seeds move it.
# Reporting differences without that reference invites reading noise as effect.

def agg(rows, key="accuracy"):
    v = np.array([r[key] for r in rows], dtype=float)
    return float(v.mean()), (float(v.std(ddof=1)) if len(v) > 1 else 0.0)

base_label = [a[1] for a in arms if a[0] == "baseline"][0]
base_acc, base_sd = agg(sweep_results[base_label])
base_rec, _ = agg(sweep_results[base_label], "recall")
base_fpr, _ = agg(sweep_results[base_label], "fpr")

print("BASELINE  %s" % base_label)
print("  accuracy %.4f +/- %.4f   recall %.4f   fpr %.4f"
      % (base_acc, base_sd, base_rec, base_fpr))
print("  seed spread is the reference: an arm matters only if it exceeds it.")
print("")

SWEEP_TITLES = {
    "split":      "SPLIT RATIO (R3-Q7)",
    "patience":   "EARLY-STOPPING PATIENCE (R3-Q9)",
    "checkpoint": "CHECKPOINT CRITERION (R3-Q8)",
    "module1":    "MODULE 1 INGESTION (R3 report)",
}

summary = {}
for sweep in ["split", "patience", "checkpoint", "module1"]:
    labels = [base_label] + [a[1] for a in arms if a[0] == sweep]
    print(SWEEP_TITLES[sweep])
    print("  configuration              accuracy          recall     fpr      d(acc)")
    rows_out = {}
    for lab in labels:
        if lab not in sweep_results: continue
        a, sd = agg(sweep_results[lab])
        r, _ = agg(sweep_results[lab], "recall")
        f, _ = agg(sweep_results[lab], "fpr")
        d = a - base_acc
        flag = "" if abs(d) <= base_sd else ("  <- beyond seed spread")
        print("  %-26s %.4f +/- %.4f   %.4f   %.4f   %+.4f%s"
              % (lab, a, sd, r, f, d, flag))
        rows_out[lab] = {"accuracy": a, "accuracy_sd": sd, "recall": r, "fpr": f,
                         "delta_vs_baseline": d,
                         "exceeds_seed_spread": bool(abs(d) > base_sd)}
    summary[sweep] = rows_out
    print("")

print("=" * 72)
print("WHAT THE SWEEPS SUPPORT")
print("=" * 72)
for sweep, title in SWEEP_TITLES.items():
    moved = [l for l, v in summary[sweep].items()
             if l != base_label and v["exceeds_seed_spread"]]
    print("  %-34s %s" % (title.split(" (")[0],
                          "sensitive: %s" % ", ".join(moved) if moved
                          else "no effect beyond seed variance"))


## Persist

In [ ]:
# ── Cell 7 · Save ─────────────────────────────────────────────────────────────
sweep_out = {
    "notebook": "11_ablation_sensitivity",
    "seed": SEED,
    "protocol": {
        "design": "one factor varied at a time; all others held at the NB05 baseline",
        "baseline": CONFIG["BASE"],
        "seeds_per_arm": CONFIG["SWEEP_SEEDS"],
        "reference": ("the baseline's own seed spread. An arm is reported as "
                      "sensitive only where its change exceeds that spread, so "
                      "seed noise is not read as an effect."),
        "resplitting": ("Changing the ratio re-partitions the data, and NB02 "
                        "assigns payload templates per partition. Dropping "
                        "mismatched chunks lost 23% and lost different amounts "
                        "per ratio, confounding the comparison with sample size. "
                        "The dataset is instead regenerated per ratio using the "
                        "recorded injection spans, so every chunk is retained and "
                        "both group and template disjointness are re-verified."),
    },
    "questions": {
        "R3-Q7": "justification for 70/15/15 and comparison against alternatives",
        "R3-Q8": "checkpoint on validation accuracy while early stopping on loss",
        "R3-Q9": "rationale for patience=3 and sensitivity to it",
        "R3-report": "whether decoding before detection improves detection",
    },
    "results": summary,
    "per_run": {k: v for k, v in sweep_results.items()},
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}
with open(DIRS["results"] / "sweep_results.json", "w") as f:
    json.dump(sweep_out, f, indent=2)

if IN_COLAB and DRIVE_ROOT:
    dst = DRIVE_ROOT / "data" / "results"; dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DIRS["results"] / "sweep_results.json", dst / "sweep_results.json")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    subprocess.run(["git","-C",str(ROOT),"add","-f",
                    "data/results/sweep_results.json"], check=False)
    st = subprocess.run(["git","-C",str(ROOT),"status","--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        subprocess.run(["git","-C",str(ROOT),"commit","-q","-m",
                        "NB11: sensitivity sweeps for R3-Q7/Q8/Q9 and Module 1 ablation"],
                       check=False)
        pr = subprocess.run(["git","-C",str(ROOT),"push","-q","origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])

print("written: sweep_results.json")


## Summary

**Paste the output below into the chat.**

The Module 1 ablation is the one I will read first: it is the only arm that tests a design
claim of the framework rather than a training hyperparameter.

In [ ]:
# ── Cell 8 · NB11 SUMMARY — paste this output ─────────────────────────────────
print("=" * 72)
print("NB11 — ABLATION AND SENSITIVITY SWEEPS")
print("=" * 72)
print("design : one factor at a time, others held at the NB05 baseline")
print("seeds  : %s per arm" % CONFIG["SWEEP_SEEDS"])
print("runs   : %d" % sum(len(v) for v in sweep_results.values()))
print("")
print("BASELINE  %s" % base_label)
print("  accuracy %.4f +/- %.4f   recall %.4f   fpr %.4f"
      % (base_acc, base_sd, base_rec, base_fpr))
print("  the seed spread (+/- %.4f) is the yardstick for every arm below" % base_sd)
for sweep in ["split", "patience", "checkpoint", "module1"]:
    print("")
    print(SWEEP_TITLES[sweep])
    print("  configuration              accuracy          recall     fpr      d(acc)")
    for lab, v in summary[sweep].items():
        flag = "  <- beyond seed spread" if v["exceeds_seed_spread"] and lab != base_label else ""
        print("  %-26s %.4f +/- %.4f   %.4f   %.4f   %+.4f%s"
              % (lab, v["accuracy"], v["accuracy_sd"], v["recall"], v["fpr"],
                 v["delta_vs_baseline"], flag))
print("")
print("VERDICT")
for sweep, title in SWEEP_TITLES.items():
    moved = [l for l, v in summary[sweep].items()
             if l != base_label and v["exceeds_seed_spread"]]
    print("  %-30s %s" % (title.split(" (")[0],
                          "sensitive: %s" % ", ".join(moved) if moved
                          else "no effect beyond seed variance"))
print("=" * 72)
print("NB11 COMPLETE — ready for NB12 (aggregation, all numbers reconciled)")
print("=" * 72)
